# Question Analysis V2

*June 12 2025*

This notebook:

1. **Loads raw data**
2. **Drops or renames junk columns** to keep only useful fields.
3. **Normalizes meta-tags**

   * text trimmed & capitalized,
   * “true/false” → booleans,
   * lists/JSON flattened.
4. **Adds helper columns** .
5. **Saves a clean,  dataset** for downstream analysis.



In [1]:
import os
import pandas as pd

## Loading and Data Inspection


In [2]:
data_path = r"data/QuestionDataLegacy.csv"
df = pd.read_csv(data_path)
print(df.columns)
df.head()

Index(['Question Title', 'Unnamed: 0', 'info.json', 'question.html',
       'server.js', 'solution.html', 'server.py', 'properties.js',
       'info1.json', 'server_trap.js', 'server1.py', 'server2.py', 'test1.py',
       'server3.py', '.DS_Store', 'question', 'question_embedding'],
      dtype='object')


,Question Title,Unnamed: 0,info.json,question.html,server.js,solution.html,server.py,properties.js,info1.json,server_trap.js,server1.py,server2.py,test1.py,server3.py,.DS_Store,question,question_embedding
0,3dMoment1,0.0,"{\r\n ""uuid"": ""8abcd91d6-64d1-4e89-b72d-ad2...",<pl-question-panel>\r\n <pl-figure file-nam...,const math = require('mathjs');\r\n// const ma...,<pl-solution-panel>\r\n <pl-figure file-nam...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,determine the moment about point o in ($n \cdo...,"[-0.014649350196123123, -0.019685911014676094,..."
1,3dMoment2,1.0,"{\r\n ""uuid"": ""8abcd91d6-64d1-4e89-b72d-ad2...",<pl-question-panel>\r\n <pl-figure file-nam...,const math = require('mathjs');\r\n// const ma...,<pl-solution-panel>\r\n <pl-figure file-nam...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Refer to the diagram labeled ""3dMoment2.png"". ...","[-0.01250634528696537, 0.0026884342078119516, ..."
2,3dMoment3,2.0,"{\r\n ""uuid"": ""8abcd91d6-64d1-4e89-b72d-ad2...",<pl-question-panel>\r\n <pl-figure file-nam...,const math = require('mathjs');\r\n// const ma...,<pl-solution-panel>\r\n <pl-figure file-nam...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,determine the resultant moment produced by the...,"[-0.02083856426179409, -0.010903744958341122, ..."
3,3dMoment4,3.0,"{\r\n ""uuid"": ""8abcd91d6-64d1-4e89-b72d-ad2...",<pl-question-panel>\r\n <pl-figure file-nam...,const math = require('mathjs');\r\n// const ma...,<pl-solution-panel>\r\n <pl-figure file-nam...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,determine the resultant moments produced by th...,"[-0.026474930346012115, -0.008942083455622196,..."
4,Equilibrium1,4.0,"{\r\n ""uuid"": ""8abcd91d6-64d1-4e89-b72d-ad2...","<pl-question-panel>\r\n<pl-figure file-name=""3...",const math = require('mathjs');\r\n// const ma...,<pl-solution-panel>\r\n <pl-figure file-nam...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,compute the missing force vector that will all...,"[-0.006892481818795204, -0.0054155210964381695..."


## Cleaning Data

The following section just keeps the columns of interest


In [3]:
columns_to_keep = [
    "Question Title",
    "info.json",
    "question.html",
    "server.js",
    "solution.html",
    "server.py",
    "properties.js",
    "question",
]
filtered_df = df[columns_to_keep]
filtered_df.head()

,Question Title,info.json,question.html,server.js,solution.html,server.py,properties.js,question
0,3dMoment1,"{\r\n ""uuid"": ""8abcd91d6-64d1-4e89-b72d-ad2...",<pl-question-panel>\r\n <pl-figure file-nam...,const math = require('mathjs');\r\n// const ma...,<pl-solution-panel>\r\n <pl-figure file-nam...,NaN,NaN,determine the moment about point o in ($n \cdo...
1,3dMoment2,"{\r\n ""uuid"": ""8abcd91d6-64d1-4e89-b72d-ad2...",<pl-question-panel>\r\n <pl-figure file-nam...,const math = require('mathjs');\r\n// const ma...,<pl-solution-panel>\r\n <pl-figure file-nam...,NaN,NaN,"Refer to the diagram labeled ""3dMoment2.png"". ..."
2,3dMoment3,"{\r\n ""uuid"": ""8abcd91d6-64d1-4e89-b72d-ad2...",<pl-question-panel>\r\n <pl-figure file-nam...,const math = require('mathjs');\r\n// const ma...,<pl-solution-panel>\r\n <pl-figure file-nam...,NaN,NaN,determine the resultant moment produced by the...
3,3dMoment4,"{\r\n ""uuid"": ""8abcd91d6-64d1-4e89-b72d-ad2...",<pl-question-panel>\r\n <pl-figure file-nam...,const math = require('mathjs');\r\n// const ma...,<pl-solution-panel>\r\n <pl-figure file-nam...,NaN,NaN,determine the resultant moments produced by th...
4,Equilibrium1,"{\r\n ""uuid"": ""8abcd91d6-64d1-4e89-b72d-ad2...","<pl-question-panel>\r\n<pl-figure file-name=""3...",const math = require('mathjs');\r\n// const ma...,<pl-solution-panel>\r\n <pl-figure file-nam...,NaN,NaN,compute the missing force vector that will all...


This section processes the info.json column and converts the json structure into dataframe columns

In [4]:
import json
import numpy as np

def process_info(raw: str):
    try:
        data: dict = json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"Could not parse {e}")
        return {}

    out = {}
    for k, v in data.items():
        if isinstance(v, str):
            out[k] = v.strip() if k == "createdBy" else v.strip().capitalize()
        # Handle list of str and list of dicts
        elif isinstance(v, list):
            if all(isinstance(x, str) for x in v):
                out[k] = ",".join([x.strip().capitalize() for x in v])
            else:
                out[k] = v
        elif v is None:
            out[k] = np.nan
        else:
            out[k] = v
    return out


meta_series = filtered_df["info.json"].apply(process_info)
print(type(meta_series))
meta_df = pd.json_normalize(meta_series)
df2 = filtered_df.reset_index(drop=True).join(meta_df)
df2

<class 'pandas.core.series.Series'>


,Question Title,info.json,question.html,server.js,solution.html,server.py,properties.js,question,uuid,title,...,isAdaptive,createdBy,qType,nSteps,updatedBy,difficulty,codelang,resources,stepType,dificulty
0,3dMoment1,"{\r\n ""uuid"": ""8abcd91d6-64d1-4e89-b72d-ad2...",<pl-question-panel>\r\n <pl-figure file-nam...,const math = require('mathjs');\r\n// const ma...,<pl-solution-panel>\r\n <pl-figure file-nam...,NaN,NaN,determine the moment about point o in ($n \cdo...,8abcd91d6-64d1-4e89-b72d-ad2133f25b2f,3d moments,...,True,epere194@ucr.edu,Num,1,,1.0,Javascript,NaN,NaN,NaN
1,3dMoment2,"{\r\n ""uuid"": ""8abcd91d6-64d1-4e89-b72d-ad2...",<pl-question-panel>\r\n <pl-figure file-nam...,const math = require('mathjs');\r\n// const ma...,<pl-solution-panel>\r\n <pl-figure file-nam...,NaN,NaN,"Refer to the diagram labeled ""3dMoment2.png"". ...",8abcd91d6-64d1-4e89-b72d-ad2133f25b2f,3d moments,...,True,epere194@ucr.edu,Num,1,,1.0,Javascript,NaN,NaN,NaN
2,3dMoment3,"{\r\n ""uuid"": ""8abcd91d6-64d1-4e89-b72d-ad2...",<pl-question-panel>\r\n <pl-figure file-nam...,const math = require('mathjs');\r\n// const ma...,<pl-solution-panel>\r\n <pl-figure file-nam...,NaN,NaN,determine the resultant moment produced by the...,8abcd91d6-64d1-4e89-b72d-ad2133f25b2f,3d moments,...,True,epere194@ucr.edu,Num,1,,1.0,Javascript,NaN,NaN,NaN
3,3dMoment4,"{\r\n ""uuid"": ""8abcd91d6-64d1-4e89-b72d-ad2...",<pl-question-panel>\r\n <pl-figure file-nam...,const math = require('mathjs');\r\n// const ma...,<pl-solution-panel>\r\n <pl-figure file-nam...,NaN,NaN,determine the resultant moments produced by th...,8abcd91d6-64d1-4e89-b72d-ad2133f25b2f,3d moments,...,True,epere194@ucr.edu,Num,1,,1.0,Javascript,NaN,NaN,NaN
4,Equilibrium1,"{\r\n ""uuid"": ""8abcd91d6-64d1-4e89-b72d-ad2...","<pl-question-panel>\r\n<pl-figure file-name=""3...",const math = require('mathjs');\r\n// const ma...,<pl-solution-panel>\r\n <pl-figure file-nam...,NaN,NaN,compute the missing force vector that will all...,8abcd91d6-64d1-4e89-b72d-ad2133f25b2f,3d statics,...,True,epere194@ucr.edu,Num,1,,1.0,Javascript,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258,preclass_2.1.6_Truck Acceleration Problem,"{""uuid"": ""d7f7a3e6-8b3d-4e3b-9c3a-6f3d7a8b3d4e...",<pl-question-panel>\r\n <p> A truck traveling...,const math = require('mathjs');\r\n\r\n\r\n\r\...,"<pl-solution-panel>\r\n <pl-hint level=""1"" ...",NaN,NaN,a truck traveling at {{params.v}} {{params.uni...,D7f7a3e6-8b3d-4e3b-9c3a-6f3d7a8b3d4e,Truck acceleration problem,...,True,lberm007@ucr.edu,Num,1,,NaN,Javascript,NaN,NaN,1.0
259,preclass_2.1.7_Vertical Ball Throw,"{\r\n ""uuid"": ""d6f8b8a7-3c3e-4e3d-9b3a-6f3c...",<pl-question-panel>\r\n <p> A ball is thrown ...,const math = require('mathjs');\r\n\r\nconst g...,"<pl-solution-panel>\r\n <pl-hint level=""1"" ...",NaN,NaN,input: consider a ball thrown vertically upwar...,D6f8b8a7-3c3e-4e3d-9b3a-6f3c4d2e1a2b,Vertical ball throw,...,True,lberm007@ucr.edu,Num,1,,NaN,Javascript,NaN,NaN,1.0
260,preclass_2.1.8_Cannon Ball Velocity,"{""uuid"": ""a1b2c3d4-e5f6-g7h8-i9j0-k1l2m3n4o5p""...",<pl-question-panel>\r\n <p> A cannon fires a ...,const math = require('mathjs');\r\n\r\nconst g...,"<pl-solution-panel>\r\n <pl-hint level=""1"" ...",NaN,NaN,a cannon fires a cannon ball at an angle of 45...,A1b2c3d4-e5f6-g7h8-i9j0-k1l2m3n4o5p,Cannon ball velocity,...,True,lberm007@ucr.edu,Num,1,,NaN,Javascript,NaN,NaN,1.0
261,preclass_2.1.9_Maximum Height of Cannon Ball,"{""uuid"": ""a1b2c3d4-e5f6-g7h8-i9j0-k1l2m3n4o5p""...",<pl-question-panel>\r\n <p> A cannon fires a ...,const math = require('mathjs');\r\n\r\nconst g...,"<pl-solution-panel>\r\n <pl-hint level=""1"" ...",NaN,NaN,a cannon fires a cannon ball at an angle {{par...,A1b2c3d4-e5f6-g7h8-i9j0-k1l2m3n4o5p,Maximum height of cannon ball,...,True,lberm007@ucr.edu,Num,1,,NaN,Javascript,NaN,NaN,1.0


## Extract Tag and Topics

In [5]:
from collections import defaultdict
import pandas as pd
import numpy as np

df = df2.copy()
labels = ["topic", "tags"]
def split_cell(cell):
    if pd.isna(cell):
        return []
    return [x.strip() for x in str(cell).split(",") if x.strip()]


unique_labels = set()
examples = defaultdict(list)
for topic_vals, tag_vals, question in df[labels + ["question"]].itertuples(index=False):
    for label in split_cell(topic_vals) + split_cell(tag_vals):
        unique_labels.add(label)
        if len(examples[label]) < 5:

            examples[label].append(question)
print(f"{len(unique_labels)} unique labels")
print(unique_labels)  # set of all tags / topics
print(examples["Heat transfer rate"])
print(examples)

with open("data/unique_tags_topics_data.json" ,"w", encoding="utf-8") as f:
    json.dump(examples, f, indent=2, ensure_ascii=False)

153 unique labels
{'Power', 'Matrix', 'Addition', 'Distance', 'Len', 'Reversibility', 'Subtraction', 'Numbers', '3d', 'Heat transfer rate', 'Heat exhanger', 'Force', 'Tension', 'Pressure', 'Multiplication', 'Work', 'Add', 'Centroid', 'Mechanics of materials', 'Heat transfer', 'Moments', 'Algorithm', 'Pro', 'Motion', 'Horizontal motion', 'Cross sectional area', 'Conduction', 'Gravity', 'Heat input', 'T', 'Area', 'Fluid', 'K', 'Difference', 'Ideal gas law', 'Cartesian coordinates', 'Trusses', 'Cop', 'Allowable stress', 'Thrust', 'Heat', 'Time calculation', '3d rigid bodies', 'V3', 'Ideal gas', 'Distance-time graph', 'Distributed forces', 'Density', 'Linear algebra', 'Differential control volume', 'Secondlaw', 'Algebra', 'Tri', 'For2', '2 cycles', 'Calculus', 'Specificheat', 'Bridge construction', 'Math', 'Energy', 'Members', 'Constant acceleration', 'Prismatic bars', 'Proportional limit', 'V2', 'Acceleration', 'Isentropic', 'Heat flux', 'Derivatives', 'Bicycle race', 'Physics', 'Sky divi

In [6]:
row_num, col_num = df.shape
adaptive_questions_count = df2["isAdaptive"].astype(bool).sum()
print(adaptive_questions_count)
print(
    f"""The number of adaptive questions is {adaptive_questions_count} \n while non adaptive questions are {row_num-adaptive_questions_count}
"""
)
py_filtered = df2[(df2["isAdaptive"].astype(bool)) & (df2["server.py"].notna())]
js_filtered = df2[(df2["isAdaptive"].astype(bool)) & (df2["server.js"].notna())]

263
The number of adaptive questions is 263 
 while non adaptive questions are 0



## Available Tags

The following are the available tags used in most questions


In [7]:
import re

pattern = r"</pl-[a-z\-]+>"
unique_tags = set()
filtered = df2[~df2["isAdaptive"].isna()]
for text in filtered["question.html"]:
    if isinstance(text, str):
        result = re.findall(pattern, text)
        unique_tags.update(result)

## Save


In [8]:
df2.to_csv("data/QuestionDataV2_06122025.csv")